In [0]:
# Databricks notebook source
# GOLD LAYER: Driver Season Statistics
# Purpose: Full-season aggregated profile per driver
# Think: the stats card shown on F1's official website per driver

from pyspark.sql.functions import (
    col, sum as spark_sum, avg, count, when, lit,
    round as spark_round, max as spark_max, min as spark_min,
    stddev, countDistinct, current_timestamp,
    collect_list, sort_array, dense_rank, row_number
)
from pyspark.sql.window import Window

# ============ 1. READ FROM SILVER ============
fact = spark.read.table("f1_data.silver.fact_results")

race_results = fact.filter(col("session_type").isin("Race", "Sprint"))

# ============ 2. AGGREGATE SEASON STATS ============
season_stats = race_results.groupBy(
    "driver_number", "year"
    # "driver_name",
    # "driver_first_name",
    # "driver_last_name",
    # "driver_acronym",
    # "driver_country_code",
    # "team_name", # CRITICAL ISSUE! Mid season, ferrari renamed to ferrari HP
    # "team_colour",
).agg(
    # ── POINTS ──
    spark_sum("points").alias("total_points"),
    avg("points").alias("avg_points_per_race_raw"),
    spark_max("points").alias("best_single_race_points"),
    stddev("points").alias("points_std_dev"),

    # ── RACE COUNTS ──
    count("session_key").alias("races_entered"),
    spark_sum(when(col("dnf") == False, 1).otherwise(0)).alias("races_finished"),
    spark_sum(when(col("points") > 0, 1).otherwise(0)).alias("points_finishes"),

    # ── POSITIONS ──
    spark_sum(when(col("position") == 1, 1).otherwise(0)).alias("wins"),
    spark_sum(when(col("position") == 2, 1).otherwise(0)).alias("second_places"),
    spark_sum(when(col("position") == 3, 1).otherwise(0)).alias("third_places"),
    spark_sum(when(col("position") <= 3, 1).otherwise(0)).alias("podiums"),
    spark_sum(when(col("position") <= 5, 1).otherwise(0)).alias("top5_finishes"),
    spark_sum(when(col("position") <= 10, 1).otherwise(0)).alias("top10_finishes"),
    avg("position").alias("avg_finish_position_raw"),
    spark_min("position").alias("best_finish"),
    spark_max("position").alias("worst_finish"),

    # ── RELIABILITY ──
    spark_sum(when(col("dnf") == True, 1).otherwise(0)).alias("dnfs"),
    spark_sum(when(col("dns") == True, 1).otherwise(0)).alias("dns_count"),
    spark_sum(when(col("dsq") == True, 1).otherwise(0)).alias("dsqs"),

    # ── LAPS ──
    spark_sum("number_of_laps").alias("total_laps_completed"),
    avg("number_of_laps").alias("avg_laps_per_race_raw"),

    # ── CIRCUITS ──
    countDistinct("meeting_key").alias("circuits_raced"),
    countDistinct("race_country").alias("countries_raced"),

    # # ── BEAT TEAMMATE ──
    # spark_sum(
    #     when(col("beat_teammate") == True, 1).otherwise(0)
    # ).alias("times_beat_teammate")
) \
.withColumn("avg_points_per_race",
    spark_round(col("avg_points_per_race_raw"), 3)
) \
.withColumn("avg_finish_position",
    spark_round(col("avg_finish_position_raw"), 2)
) \
.withColumn("avg_laps_per_race",
    spark_round(col("avg_laps_per_race_raw"), 1)
) \
.withColumn("points_std_dev",
    spark_round(col("points_std_dev"), 3)
) \
.drop("avg_points_per_race_raw", "avg_finish_position_raw", "avg_laps_per_race_raw")

# ============ 2b. JOIN DRIVER LABELS BACK ============
# Pull the most recent label per driver per year from dim_drivers
# race_results already has team_name, driver_name, etc joined in.
# Pick the row with the highest meeting_key per driver = end-of-season label.
label_window = Window \
    .partitionBy("driver_number", "year") \
    .orderBy(col("meeting_key").desc())

driver_labels = race_results \
    .withColumn("rn", row_number().over(label_window)) \
    .filter(col("rn") == 1) \
    .select(
        "driver_number",
        "year",
        "driver_name",
        "driver_first_name",
        "driver_last_name",
        "driver_acronym",
        "driver_country_code",
        "team_name",
        "team_colour"
    )

season_stats = season_stats.join(
    driver_labels,
    on=["driver_number", "year"],
    how="left"
)

# ============ 3. RATE METRICS ============
season_stats = season_stats \
    .withColumn("win_rate",
        spark_round(col("wins") / col("races_entered") * 100, 2)
    ) \
    .withColumn("podium_rate",
        spark_round(col("podiums") / col("races_entered") * 100, 2)
    ) \
    .withColumn("points_finish_rate",
        spark_round(col("points_finishes") / col("races_entered") * 100, 2)
    ) \
    .withColumn("dnf_rate",
        spark_round(col("dnfs") / col("races_entered") * 100, 2)
    ) \
    .withColumn("reliability_score",
        # 100 = perfect reliability, 0 = never finished
        spark_round(
            (col("races_finished") / col("races_entered")) * 100, 2
        )
    ) \
    # .withColumn("teammate_beat_rate",
    #     spark_round(col("times_beat_teammate") / col("races_entered") * 100, 2)
    # )

# ============ 4. PERFORMANCE TIERS ============
# Classify drivers into performance tiers based on average points
tier_window = Window.partitionBy("year").orderBy(col("total_points").desc())

season_stats = season_stats \
    .withColumn("championship_position",
        dense_rank().over(tier_window)
    ) \
    .withColumn("performance_tier",
        when(col("championship_position") <= 3, lit("CHAMPIONSHIP_CONTENDER"))
        .when(col("championship_position") <= 6, lit("TOP_MIDFIELD"))
        .when(col("championship_position") <= 12, lit("MIDFIELD"))
        .otherwise(lit("BACKMARKER"))
    ) \
    .withColumn("consistency_rating",
        # Low std_dev + high avg = consistent
        # Scale: A/B/C/D
        when(
            (col("avg_points_per_race") >= 15) & (col("points_std_dev") <= 8),
            lit("A - Dominant & Consistent")
        )
        .when(
            (col("avg_points_per_race") >= 10) & (col("points_std_dev") <= 12),
            lit("B - Competitive & Reliable")
        )
        .when(
            (col("avg_points_per_race") >= 5),
            lit("C - Points Scorer")
        )
        .otherwise(lit("D - Inconsistent/Backmarker"))
    )

# ============ 5. WRITE TO GOLD ============
season_stats.withColumn("gold_processed_at", current_timestamp()) \
    .withColumn("gold_source", lit("f1_data.silver.fact_results")) \
    .write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("year") \
    .saveAsTable("f1_data.gold.driver_season_stats")

print(f"✓ gold.driver_season_stats written: {season_stats.count()} records")

print("\n📊 SEASON STATS PREVIEW:")
spark.sql("""
    SELECT
        championship_position,
        driver_name,
        team_name,
        total_points,
        wins,
        podiums,
        dnfs,
        reliability_score,
        consistency_rating,
        performance_tier
    FROM f1_data.gold.driver_season_stats
    WHERE year = 2024
    ORDER BY championship_position
""").show(20, truncate=False)

✓ gold.driver_season_stats written: 23 records

📊 SEASON STATS PREVIEW:
+---------------------+----------------+---------------+------------+----+-------+----+-----------------+---------------------------+----------------------+
|championship_position|driver_name     |team_name      |total_points|wins|podiums|dnfs|reliability_score|consistency_rating         |performance_tier      |
+---------------------+----------------+---------------+------------+----+-------+----+-----------------+---------------------------+----------------------+
|1                    |Max Verstappen  |Red Bull Racing|331.0       |10  |14     |1   |95.24            |B - Competitive & Reliable |CHAMPIONSHIP_CONTENDER|
|2                    |Lando Norris    |McLaren        |279.0       |3   |12     |1   |95.24            |B - Competitive & Reliable |CHAMPIONSHIP_CONTENDER|
|3                    |Charles Leclerc |Ferrari        |245.0       |2   |10     |1   |95.24            |B - Competitive & Reliable |CHAMPIONSH

In [0]:
%sql
DESCRIBE TABLE f1_data.silver.dim_drivers

col_name,data_type,comment
driver_number,int,null
meeting_key,int,null
session_key,int,null
first_name,string,null
last_name,string,null
full_name,string,null
broadcast_name,string,null
name_acronym,string,null
team_name,string,null
team_colour,string,null
